# Lab 2.4 — Build a Semantic Cache

**Before you start:** select **Cell > Run All** to initialize the harness.

Cells marked `# ── YOUR WORK ──` are the ones you edit.

In [ ]:
# ── Harness setup (run once) ──────────────────────────────────────────────────
import sys, os, json, pathlib, time

sys.path.insert(0, '/opt/ara/lib')
from tina.client import llm_client, model_fast, model_strong
from elasticsearch import Elasticsearch

env_path = pathlib.Path('/home/elastic/env')
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            k, v = line.split('=', 1)
            os.environ.setdefault(k.strip(), v.strip())

client = llm_client()
FAST   = model_fast()
STRONG = model_strong()
es = Elasticsearch(os.environ['ES_URL'], api_key=os.environ['ES_API_KEY'], request_timeout=60)
EMBED_ID = os.environ.get('ARA_EMBED_ID', '.jina-embeddings-v5-text-small')
TRACES = pathlib.Path('/home/elastic/.traces')
TRACES.mkdir(parents=True, exist_ok=True)
print(f'Harness ready. FAST={FAST}, STRONG={STRONG}')

In [ ]:
# ── YOUR WORK ── Implement cache_lookup and cache_store ─────────────────────
import time

THRESHOLD = 0.85  # Adjust: too low → near-misses hit; too high → paraphrases miss

def cache_lookup(query_text):
    """Return cached answer if a similar query exists above threshold; else None."""
    r = es.search(index='cortex-semantic-cache', body={
        'query': {'semantic': {'field': 'query', 'query': query_text}},
        'size': 1, '_source': ['answer']
    })
    if r['hits']['hits']:
        score = r['hits']['hits'][0].get('_score', 0) or 0
        if score >= THRESHOLD:
            return r['hits']['hits'][0]['_source']['answer']
    return None

def cache_store(query_text, answer):
    """Store query + answer in the semantic cache."""
    es.index(index='cortex-semantic-cache', body={
        'query': query_text, 'query_text': query_text,
        'answer': answer, 'cached_at': int(time.time() * 1000)
    }, refresh=True)

print(f'Cache functions defined (threshold={THRESHOLD})')

In [ ]:
# Run dev sequence and save results
import json, pathlib, time
from tina.rag import rag_answer

dev_queries = [
    'What is the CTR threshold at Cortex Bank?',
    'What dollar amount triggers a Currency Transaction Report at Cortex Bank?',
    'How much cash triggers a CTR filing requirement at Cortex Bank?',
    'How long does Cortex Bank have to file a SAR after detection?',
    'What is the SAR filing deadline at Cortex Bank once suspicious activity is identified?',
]

results = []
for q in dev_queries:
    cached = cache_lookup(q)
    if cached:
        results.append({'query': q, 'outcome': 'hit'})
        print(f'  HIT: {q[:50]}')
    else:
        trace = rag_answer(es, q, 'cortex-corpus-live', completion_id='cortex-generation',
                           trace_dir='/home/elastic/.traces')
        answer = trace.get('answer', '')
        cache_store(q, answer)
        results.append({'query': q, 'outcome': 'miss'})
        print(f'  MISS: {q[:50]}')

es.indices.refresh(index='cortex-semantic-cache')
cache_count = es.count(index='cortex-semantic-cache')['count']
hits = sum(1 for r in results if r['outcome'] == 'hit')
hit_rate = hits / len(results)

pathlib.Path('/home/elastic/cache-results.json').write_text(json.dumps({
    'threshold': THRESHOLD, 'dev_results': results,
    'cache_doc_count': cache_count, 'hit_rate': hit_rate
}))
print(f'Hit rate: {hits}/{len(results)} = {hit_rate:.2f}. Select Check in the sidebar.')